# 05 — Evaluasi Model & Simulasi Inferensi End-to-End

Notebook ini mengevaluasi kedua model yang tersimpan di `models/` (Confusion Matrix + Kurva Loss) serta memverifikasi inferensi *end-to-end* terhadap input form web.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

BASE_DIR = Path("..") if Path("../models").exists() else Path(".")
sleep_model = joblib.load(BASE_DIR / "models" / "sleep_model.keras")
sleep_scaler = joblib.load(BASE_DIR / "models" / "sleep_scaler.pkl")
prod_model = joblib.load(BASE_DIR / "models" / "productivity_model.keras")
prod_scaler = joblib.load(BASE_DIR / "models" / "productivity_scaler.pkl")

In [ ]:
df_sleep = pd.read_csv(BASE_DIR / "dataset" / "processed" / "sleep_cleaned.csv")
X1 = df_sleep[["Sleep Duration", "Quality of Sleep", "Physical Activity Level", "Stress Level"]].values
y1 = df_sleep["target"].values
_, X1_test, _, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42, stratify=y1)
y1_pred = sleep_model.predict(sleep_scaler.transform(X1_test))

df_prod = pd.read_csv(BASE_DIR / "dataset" / "processed" / "productivity_cleaned.csv")
X2 = df_prod[["sleep_hours", "phone_usage_hours", "coffee_intake_mg", "exercise_minutes", "stress_level"]].values
y2 = df_prod["productivity_class"].values
_, X2_test, _, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)
y2_pred = prod_model.predict(prod_scaler.transform(X2_test))

print(f"Model 1 (Insomnia) Accuracy     : {accuracy_score(y1_test, y1_pred):.4f}")
print(classification_report(y1_test, y1_pred, target_names=["Normal", "Insomnia"]))
print(f"Model 2 (Produktivitas) Accuracy: {accuracy_score(y2_test, y2_pred):.4f}")
print(classification_report(y2_test, y2_pred, target_names=["Rendah", "Sedang", "Tinggi"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.heatmap(confusion_matrix(y1_test, y1_pred), annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Normal", "Insomnia"], yticklabels=["Normal", "Insomnia"])
axes[0].set_title("Confusion Matrix — Insomnia")

sns.heatmap(confusion_matrix(y2_test, y2_pred), annot=True, fmt="d", cmap="Greens", ax=axes[1],
            xticklabels=["Rendah", "Sedang", "Tinggi"], yticklabels=["Rendah", "Sedang", "Tinggi"])
axes[1].set_title("Confusion Matrix — Produktivitas")

axes[2].plot(sleep_model.loss_curve_, label="Insomnia Loss")
axes[2].plot(prod_model.loss_curve_, label="Produktivitas Loss")
axes[2].set_title("Training Loss Curve")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
plt.show()